# 10 — Uniform downward pass and complete FMM validation

This notebook uses the compiled C++ `UniformFmm`; it does **not** reproduce the traversal in Python. Part A follows one selected target's far-field route (`list2` M2L → ancestor locals → L2L → leaf L2P) and its direct `list1` P2P near field. Arrows between box centres show the **translation route**, not the orientation of any local-expansion coefficient vector.

Colours: orange = M2L source boxes/routes, purple = L2L target path, green = L2P, blue = list1/P2P, red star = selected target, grey = inactive boxes.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display

import cdfmm
from examples.notebooks.example_utils import (
    direct_fields, draw_box_3d, error_metrics, relative_error, vec3_to_array,
)

SEED = 202603
rng = np.random.default_rng(SEED)


## Part A — trace the downward route

The small example uses 24 independent sources and 9 independent targets, depth 3, and expansion order 4. There are no source/target self identities. The evaluator first completes its C++ traversal so multipoles and locals exposed below are the actual computed state.


In [ ]:
source_positions = rng.uniform(-0.92, 0.92, size=(24, 3))
target_positions = rng.uniform(-0.92, 0.92, size=(9, 3))
moments = rng.normal(size=(24, 3))

options = cdfmm.UniformFmmOptions()
options.expansion_order = 4
options.tree.max_level = 3
options.tree.root_centre = cdfmm.Vec3(0.0, 0.0, 0.0)
options.tree.root_half_width = 1.0
fmm = cdfmm.UniformFmm(source_positions, target_positions, options)
small_result = fmm.evaluate(moments, output="both")
print(f"seed={SEED}, sources=24, targets=9, depth=3, p=4, independent sets")


In [ ]:
def draw_target_route(original_target_index):
    tree = fmm.tree
    sorted_target_index = tree.target_inverse_permutation[original_target_index]
    leaf_index = tree.leaf_index_for_target(sorted_target_index)
    nodes = tree.nodes
    path = []
    node_index_on_path = leaf_index
    while node_index_on_path >= 0:
        path.append(node_index_on_path)
        node_index_on_path = nodes[node_index_on_path].parent
    path.reverse()

    figure = plt.figure(figsize=(10, 8))
    axes = figure.add_subplot(111, projection="3d")
    for node in nodes:
        if node.level == tree.leaf_level:
            draw_box_3d(axes, vec3_to_array(node.centre), node.half_width,
                        colour="0.8", linewidth=0.25, alpha=0.18)

    # L2L follows the purple ancestor path towards the occupied target leaf.
    for parent_index, child_index in zip(path[:-1], path[1:]):
        parent = vec3_to_array(nodes[parent_index].centre)
        child = vec3_to_array(nodes[child_index].centre)
        delta = child - parent
        axes.quiver(*parent, *delta, color="tab:purple", linewidth=2,
                    arrow_length_ratio=0.12, label="L2L" if parent_index == path[0] else None)
        draw_box_3d(axes, child, nodes[child_index].half_width,
                    colour="tab:purple", linewidth=1.5, alpha=0.8)

    # Each path node receives M2L information from its own orange list2 boxes.
    labelled_m2l = False
    for target_node_index in path[1:]:
        target_node = nodes[target_node_index]
        target_centre = vec3_to_array(target_node.centre)
        for source_node_index in target_node.list2:
            source_node = nodes[source_node_index]
            if source_node.source_count == 0:
                continue
            source_centre = vec3_to_array(source_node.centre)
            delta = target_centre - source_centre
            axes.quiver(*source_centre, *delta, color="tab:orange", alpha=0.55,
                        arrow_length_ratio=0.08,
                        label="M2L" if not labelled_m2l else None)
            draw_box_3d(axes, source_centre, source_node.half_width,
                        colour="tab:orange", linewidth=0.8, alpha=0.5)
            labelled_m2l = True

    leaf = nodes[leaf_index]
    for near_index in leaf.list1:
        near = nodes[near_index]
        if near.source_count:
            draw_box_3d(axes, vec3_to_array(near.centre), near.half_width,
                        colour="tab:blue", linewidth=1.2, alpha=0.8,
                        label="list1 / P2P" if near_index == leaf.list1[0] else None)

    target = target_positions[original_target_index]
    leaf_centre = vec3_to_array(leaf.centre)
    axes.plot(*target, marker="*", markersize=16, color="tab:red", label="selected target")
    axes.quiver(*leaf_centre, *(target - leaf_centre), color="tab:green",
                linewidth=3, arrow_length_ratio=0.25, label="L2P")
    axes.scatter(source_positions[:, 0], source_positions[:, 1], source_positions[:, 2],
                 s=10, color="black", alpha=0.35)
    axes.set(xlabel="x", ylabel="y", zlabel="z",
             title=f"Downward and near-field routes for target {original_target_index}")
    axes.legend(loc="upper left")
    plt.show()

selector = widgets.IntSlider(value=0, min=0, max=len(target_positions)-1,
                             step=1, description="Target")
display(widgets.interactive_output(draw_target_route, {"original_target_index": selector}), selector)


## Part B — ensemble accuracy

This deterministic study uses 120 independent sources and 100 independent targets, tree depth 2, seed 202603, and no self exclusions. It compares the compiled complete FMM with the compiled direct P2P helper and then varies expansion order without tuning the particle cloud.


In [ ]:
ensemble_rng = np.random.default_rng(SEED)
source_count, target_count, depth = 120, 100, 2
ensemble_sources = ensemble_rng.uniform(-0.95, 0.95, size=(source_count, 3))
ensemble_targets = ensemble_rng.uniform(-0.95, 0.95, size=(target_count, 3))
ensemble_moments = ensemble_rng.normal(size=(source_count, 3))
reference_fields = direct_fields(ensemble_targets, ensemble_sources, ensemble_moments)

study = []
fields_by_order = {}
for order in (2, 3, 4):
    study_options = cdfmm.UniformFmmOptions()
    study_options.expansion_order = order
    study_options.tree.max_level = depth
    study_options.tree.root_centre = cdfmm.Vec3(0.0, 0.0, 0.0)
    study_options.tree.root_half_width = 1.0
    evaluator = cdfmm.UniformFmm(ensemble_sources, ensemble_targets, study_options)
    fields = evaluator.evaluate(ensemble_moments)["H"]
    fields_by_order[order] = fields
    study.append((order, error_metrics(fields, reference_fields)))

for order, metrics in study:
    print(f"p={order}: mean={metrics['mean']:.4e}, RMS={metrics['rms']:.4e}, "
          f"maximum={metrics['maximum']:.4e}")


In [ ]:
chosen_order = 4
chosen_fields = fields_by_order[chosen_order]
point_errors = relative_error(chosen_fields, reference_fields)
figure, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].semilogy(np.arange(target_count), point_errors, ".", color="tab:red")
axes[0].set(xlabel="Target index", ylabel="Relative field error", title="Pointwise error")
axes[0].grid(alpha=0.25)
axes[1].scatter(reference_fields[:, 0], chosen_fields[:, 0], s=12, alpha=0.65)
limits = np.array([axes[1].get_xlim(), axes[1].get_ylim()])
lo, hi = limits.min(), limits.max()
axes[1].plot([lo, hi], [lo, hi], "k--", linewidth=1)
axes[1].set(xlabel="Direct H_x", ylabel="FMM H_x", title="Direct versus FMM")
orders = [item[0] for item in study]
rms = [item[1]["rms"] for item in study]
maximum = [item[1]["maximum"] for item in study]
axes[2].semilogy(orders, rms, "o-", label="RMS")
axes[2].semilogy(orders, maximum, "s--", label="maximum")
axes[2].set(xlabel="Expansion order p", ylabel="Relative error", title="Order study")
axes[2].legend()
axes[2].grid(alpha=0.25)
figure.tight_layout()


The near/far split is exact at the box-list level: only `list1` pairs use P2P, while all well-separated `list2` information reaches a target through M2L, inherited L2L translations, and final L2P. Increasing `p` reduces truncation error; it does not alter that partition.
